# 👟 ARShoe M1v2 Training & Validation on Google Colab

This notebook executes **Step 5 (Train M1v2 from scratch)** and **Step 6 (Validation Evaluation)** using the combined dataset (`shuffled_v3 + shuffled_v4`).

### Instructions:
1. In Colab, go to **Runtime > Change runtime type** and ensure **T4 GPU** or **A100 GPU** is selected.
2. Upload `shoes_vto_code.zip` and `merged_v3_v4.zip` into `/content/`.
3. Run the cells below in order.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Unpack code and dataset
!unzip -q -o /content/shoes_vto_code.zip -d /content/
!unzip -q -o /content/merged_v3_v4.zip -d /content/dataset/

# Install dependencies
!pip install -q thop albumentations opencv-python matplotlib

print("✅ Setup complete!")

In [ ]:
# Cell 3: Verify M1v2 Model & Directory Structure
import sys
sys.path.insert(0, '/content/shoes_vto')
from src.models.arshoe_m1v2 import ARShoeM1v2

model = ARShoeM1v2()
total_p, breakdown = model.count_parameters()
print(f"ARShoe M1v2 Total Parameters: {total_p:,}")
print(f"Breakdown: {breakdown}")

# Verify dataset counts
import os
print("Train images:", len(os.listdir('/content/dataset/merged_v3_v4/train/images')))
print("Valid images:", len(os.listdir('/content/dataset/merged_v3_v4/valid/images')))
print("Test images :", len(os.listdir('/content/dataset/merged_v3_v4/test/images')))

In [ ]:
# Cell 4: Step 5 — Train ARShoe M1v2 From Scratch (100 Epochs)
%cd /content/shoes_vto

!python train_m1v2.py \
    --dataset_root /content/dataset/merged_v3_v4 \
    --output_dir outputs/m1v2_training \
    --epochs 100 \
    --batch_size 16 \
    --lr 0.0005 \
    --eta_min 0.000001 \
    --num_workers 4

In [ ]:
# Cell 5: Step 6 — Evaluate on Validation Split
%cd /content/shoes_vto

!python evaluate_m1.py \
    --checkpoint outputs/m1v2_training/checkpoints/best.pth \
    --dataset_root /content/dataset/merged_v3_v4 \
    --split valid \
    --output_dir outputs/m1v2_valid_eval \
    --num_vis 10

In [ ]:
# Cell 6: Zip Checkpoints and Evaluation Results for Download
!zip -r /content/m1v2_results.zip /content/shoes_vto/outputs/m1v2_training/checkpoints /content/shoes_vto/outputs/m1v2_valid_eval
print("🎉 Artifacts ready! Download /content/m1v2_results.zip from the Colab file explorer.")

### Step 7 — Final Test Split (DO NOT RUN YET)

> **Strict constraint**: Do not evaluate on test split until validation metrics are inspected and model is finalized.

```python
# ONLY AFTER FINAL APPROVAL:
# !python evaluate_m1.py \
#     --checkpoint outputs/m1v2_training/checkpoints/best.pth \
#     --dataset_root /content/dataset/merged_v3_v4 \
#     --split test \
#     --output_dir outputs/m1v2_test_eval \
#     --num_vis 10
```